# Create dataset

In [0]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Number of records
n =10000

# Generate dates (for trend forecasting)
dates = pd.date_range(start="2022-01-01", periods=n, freq='W')

# Features
size = np.random.normal(loc=120, scale=30, size=n).astype(int)  # square meters
bedrooms = np.random.randint(1, 5, n)
bathrooms = np.random.randint(1, 3, n)
age = np.random.randint(0, 30, n)

# Location categories
locations = ["Urban", "Suburban", "Rural"]
location = np.random.choice(locations, n)

# Location price multipliers
location_factor = {
    "Urban": 1.5,
    "Suburban": 1.2,
    "Rural": 0.8
}

# Time trend (prices increase over time)
time_trend = np.arange(n) * 500  

# Base price calculation
price = (
    size * 8000 +
    bedrooms * 50000 +
    bathrooms * 30000 -
    age * 2000 +
    time_trend +
    np.array([location_factor[l] for l in location]) * 100000 +
    np.random.normal(0, 50000, n)  # noise
)

# Create DataFrame
df = pd.DataFrame({
    "date": dates,
    "size_sqm": size,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "age": age,
    "location": location,
    "price": price.astype(int)
})

# Save dataset
df.to_csv("house_prices.csv", index=False)

df.head()

# Predictive Analytics: Forecasting House Prices

## Objective
In this session, we will:
- Understand time-based trends
- Aggregate data over time
- Visualize trends

## Key Idea
Forecasting = Using past data patterns to estimate future values

We use:

- pandas → to work with structured data (tables)
- numpy → for numerical operations
- matplotlib → for visualization

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [0]:
df = pd.read_csv("house_prices.csv")
df.head()

# Data Preparation

We always start by inspecting the dataset.

Why?
Because we need to:
- Understand columns
- Identify data types
- Check if the data makes sense

In [0]:
df.info()

We convert "date" to datetime format.

Why?
Because forecasting is time-based — Python needs to recognize this as a time variable.

Without this step:
-  We cannot group by month or plot trends correctly.

In [0]:
df["date"] = pd.to_datetime(df["date"])
df.info()

In [0]:
df = df[df["date"] <= '2028-01-31']

In [0]:
df["date"].min()

In [0]:
df["date"].max()

In [0]:
df.shape

# Feature Engineering

## Create Monthly Trend.

We aggregate data monthly.

Why?
- Daily/weekly data is noisy
- Forecasting works better on smoothed trends

This step helps us:
→ See the overall direction of prices

In [0]:
monthly_prices = df.resample("M", on ="date")["price"].mean()
monthly_prices.head()

In [0]:
monthly_prices.iloc[0]

In [0]:
monthly_prices_df = monthly_prices.reset_index()
monthly_prices_df.columns = ['date','average_price']
monthly_prices_df.head()

In [0]:
monthly_prices_df["date"].min()

## Visualize Trend

Visualization is critical.

Why?
Before forecasting, we must answer:
- Is the trend increasing?
- Is it stable?
- Is it random?

Forecasting without visualization = guessing.

In [0]:
plt.figure()
plt.plot(monthly_prices)
plt.title("Average House Prices Over Time")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

## Moving Average (Smoothing)

Moving average smooths the data.

Real-world data has noise (random fluctuations).

This helps us:
- See the true underlying trend
- Avoid overreacting to spikes

In [0]:
moving_avg = monthly_prices.rolling(window=3).mean()
moving_avg.head()

In [0]:

plt.figure()
plt.plot(monthly_prices, label="Original")
plt.plot(moving_avg, label="Moving Average")
plt.legend()
plt.show()

## Create Time Index (Critical Step)
We create a time index (0,1,2,3,...).

We cannot run regression directly on dates.

So we convert time into a numeric sequence:
- This allows us to model trends mathematically

In [0]:
monthly_df = monthly_prices.reset_index()
monthly_df["time_index"] = np.arange(len(monthly_df))
monthly_df.head()

# Build Simple Trend Model

We use `np.polyfit` to fit a straight line.

This is basic mathematics (linear regression).

Formula:
> - y = m * x + c
> - Price = slope * time + intercept

Meaning:
- slope → how fast prices are increasing
- intercept → starting point

In [0]:
coefficients = np.polyfit(monthly_df["time_index"], monthly_df["price"], 1)

slope = coefficients[0]
intercept = coefficients[1]

print("slope: ", slope)
print("Intercept: ", intercept)

## Create Forecast

We extend the time index into the future.


Forecasting = applying the same trend forward.

We assume:
-  The past trend continues into the future

In [0]:
future_period = 10

future_index = np.arange(len(monthly_df), len(monthly_df) + future_period)

#y = m*x + c
future_prices = slope * future_index + intercept

## Create Future Dates

In [0]:
last_date = monthly_df["date"].max()

future_dates = pd.date_range(start =last_date, periods=future_period+1, freq="M")[1:]

## Combine Results

In [0]:
forecast_df = pd.DataFrame({"date":future_dates,
                            "forecast_price": future_prices})
forecast_df.head()

In [0]:
forecast_df.shape

## Final Visualisation

We compare:
- Historical data (Actual)
- Predicted future (Forecast)


In [0]:
plt.figure()

#historical data
plt.plot(monthly_df["date"], monthly_df["price"], label = "Actual")

#predicted future / forecast
plt.plot(forecast_df["date"], forecast_df["forecast_price"], label = "Forecast")

plt.legend()

plt.title("House price forecast")
plt.show()

1. Is the trend increasing or decreasing?
2. Do you trust this forecast? Why or why not?
3. What assumptions are we making?
4. What could make this forecast wrong?

# Add Trend Line (Fitted Line)

The trend line is the "best fit" straight line through the data.

It represents:
- The overall direction of prices
- Ignoring short-term fluctuations

The forecast is simply:
- Extending this line into the future

In [0]:
# Create fitted values (trend line)
monthly_df["trend_line"] = slope * monthly_df["time_index"] + intercept

In [0]:
plt.figure()

# Actual data
plt.plot(monthly_df["date"], monthly_df["price"], label="Actual")

# Trend line (THIS is your straight line)
plt.plot(monthly_df["date"], monthly_df["trend_line"], label="Trend Line")

# Forecast
plt.plot(forecast_df["date"], forecast_df["forecast_price"], label="Forecast")

plt.legend()
plt.title("House Price Trend + Forecast")
plt.show()